# Presentation · Result figures (ROC comparison)

**Primary interface** for the headline ROC result figure. Calls `channel_heads.*`
only:

- `channel_heads.eval.outlet_group_holdout` — the held-out Earth test split
- `channel_heads.viz.roc_curve_panel` — the ROC panel renderer
- `channel_heads.inference.load_xgb_model` — model loading

It is **read-only**: it renders the ROC comparison inline and writes no PNGs.
(The Mars networks-coloured-by-prediction figure is produced by the batch
script.)

> Uses the baseline + regime model artifacts under `models/`. Missing variants
> are skipped.

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from channel_heads.io.paths import PROJECT_ROOT
from channel_heads.eval import outlet_group_holdout
from channel_heads.models.xgboost import load_xgb_model
from channel_heads.viz import roc_curve_panel

RESULTS = PROJECT_ROOT / "data/results"
MODELS = PROJECT_ROOT / "models"
GEOM = ["orientation_diff_deg", "headhead_dist_norm", "apex_angle_deg",
        "strahler_order_diff", "proximity_profile_norm"]
EMB = [f"emb_{i}" for i in range(4)]


def test_y_and_frame(csv):
    df = pd.read_csv(csv)
    _, test_idx = outlet_group_holdout(df)
    dt = df.iloc[test_idx]
    return dt, dt["y"].astype(int).to_numpy()


print("data dir present:", RESULTS.exists())

data dir present: True


## Baseline variants + regimes — ROC on the held-out Earth test

In [2]:
# Baseline three variants (share one test split on the production dataset)
base_csv = RESULTS / "master_dataset_v4_cnn_full.csv"
baseline_specs = [
    ("geom_only", "xgb_geom_only.json", GEOM),
    ("geom+cnn_emb", "xgb_geom_plus_cnn_emb.json", GEOM + EMB),
    ("geom+cnn_logit", "xgb_geom_plus_cnn_logit.json", GEOM + ["cnn_logit"]),
]
base_entries = []
if base_csv.exists():
    bt, y = test_y_and_frame(base_csv)
    for name, mp, feats in baseline_specs:
        path = MODELS / mp
        if path.exists() and all(f in bt.columns for f in feats):
            proba = load_xgb_model(path).predict_proba(bt[feats])[:, 1]
            base_entries.append((name, y, proba))

# Regimes (each its own dataset + test split)
reg_entries = []
for R in ["regA", "regB", "regC"]:
    csv = RESULTS / f"master_dataset_{R}_with_emb.csv"
    model = MODELS / f"xgb_geom_plus_cnn_emb_{R}.json"
    if csv.exists() and model.exists():
        dt, y = test_y_and_frame(csv)
        proba = load_xgb_model(model).predict_proba(dt[GEOM + EMB])[:, 1]
        reg_entries.append((R, y, proba))

print("baseline curves:", [e[0] for e in base_entries])
print("regime curves:", [e[0] for e in reg_entries])

baseline curves: ['geom_only', 'geom+cnn_emb', 'geom+cnn_logit']
regime curves: ['regA', 'regB', 'regC']


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
if base_entries:
    roc_curve_panel(axes[0], base_entries, "Baseline (Earth held-out test)")
if reg_entries:
    roc_curve_panel(axes[1], reg_entries, "Regimes geom+cnn_emb (Earth held-out test)")
fig.suptitle("ROC — rebuilt models on fixed rasters", fontsize=13)
fig.tight_layout()
fig

<Figure size 1200x550 with 2 Axes>

---
Full run (writes ROC + Mars-networks PNGs):

```bash
python scripts/make_result_figures.py
```